# 4.1 Failure Mode Analysis — Zero-Shot Baseline

Analyses the FM taxonomy applied to baseline (Condition 0, zero-shot) errors.  
FM1 = Parametric Override (model ignores/contradicts gold passage)  
FM2 = Reasoning Failure (model uses passage but fails to reach correct answer)

---

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

BASELINE_PATH = 'results/zero_shot/condition_0_baseline.csv'
FM_PATH       = 'results/zero_shot/failure_modes_baseline.csv'
SUBJECTS = ['CONST. LAW', 'CONTRACTS', 'CRIM. LAW', 'EVIDENCE', 'REAL PROP.', 'TORTS']

baseline = pd.read_csv(BASELINE_PATH)
baseline['is_correct'] = baseline['is_correct'].astype(bool)
fm = pd.read_csv(FM_PATH)

# Merge: FM labels only exist for baseline errors
errors = baseline[baseline['is_correct'] == False].copy()
errors = errors.merge(fm[['idx', 'failure_mode', 'justification']], on='idx', how='left')

print(f'Total questions:  {len(baseline)}')
print(f'Baseline errors:  {len(errors)} ({len(errors)/len(baseline):.1%})')
print(f'FM-classified:    {errors["failure_mode"].notna().sum()}')
print(f'Unclassified:     {errors["failure_mode"].isna().sum()}')

## 4.1.1 Classifier Choice & Limitations

In [ ]:
# --- Coverage ---
n_errors = len(errors)
n_fm1 = (errors['failure_mode'] == 'FM1').sum()
n_fm2 = (errors['failure_mode'] == 'FM2').sum()
n_unclassified = errors['failure_mode'].isna().sum()

coverage = (n_fm1 + n_fm2) / n_errors
print(f'Classification coverage: {coverage:.1%}')
print(f'  FM1 (Parametric Override): {n_fm1} ({n_fm1/n_errors:.1%})')
print(f'  FM2 (Reasoning Failure):   {n_fm2} ({n_fm2/n_errors:.1%})')
print(f'  Unclassified / NaN:        {n_unclassified} ({n_unclassified/n_errors:.1%})')

# --- Circularity note ---
print('''
Classifier: llama-3.3-70b-versatile via Groq (same model as experimental subject).
Circularity risk: the classifier is judging its own reasoning errors. FM1 errors
(parametric override) may be under-reported if the model cannot recognise when
it is ignoring its own context window.

Taxonomy coverage: FM1 and FM2 are mutually exclusive by design.
FM2 collapses two sub-types (misapplied reasoning + inference gap);
see manual taxonomy (n=20 sample) in CLAUDE.md for sub-type breakdown.
''')

# --- Justification length as a classifier quality proxy ---
errors['justification_len'] = errors['justification'].fillna('').str.len()
print('Justification length by FM type (chars):')
print(errors.groupby('failure_mode')['justification_len'].describe().round(1))

In [ ]:
# Plot: classification breakdown pie + justification length boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie
ax = axes[0]
labels = ['FM1\n(Parametric Override)', 'FM2\n(Reasoning Failure)', 'Unclassified']
sizes  = [n_fm1, n_fm2, n_unclassified]
colors = ['tomato', 'steelblue', 'lightgrey']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%', colors=colors,
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
for at in autotexts:
    at.set_fontsize(10)
ax.set_title(f'Error Classification Coverage\n(n={n_errors} baseline errors)')

# Justification length
ax = axes[1]
classified = errors[errors['failure_mode'].isin(['FM1', 'FM2'])]
groups = [classified[classified['failure_mode'] == 'FM1']['justification_len'],
          classified[classified['failure_mode'] == 'FM2']['justification_len']]
bp = ax.boxplot(groups, labels=['FM1', 'FM2'], patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2})
for patch, c in zip(bp['boxes'], ['tomato', 'steelblue']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.set_ylabel('Justification Length (chars)')
ax.set_title('Classifier Justification Length by FM Type\n(proxy for classification confidence)')

plt.suptitle('4.1.1 Classifier Coverage & Quality', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/fm_4_1_1_classifier.png', bbox_inches='tight')
plt.show()

## 4.1.2 Baseline Failure Mode Distribution

In [ ]:
# Wilson CIs on FM proportions (out of all errors)
def fm_ci(n_fm, n_total):
    lo, hi = proportion_confint(n_fm, n_total, alpha=0.05, method='wilson')
    return lo, hi

fm1_lo, fm1_hi = fm_ci(n_fm1, n_errors)
fm2_lo, fm2_hi = fm_ci(n_fm2, n_errors)

print('Baseline Error Distribution (with 95% Wilson CI):')
print(f'  FM1: {n_fm1/n_errors:.3f} [{fm1_lo:.3f}, {fm1_hi:.3f}]')
print(f'  FM2: {n_fm2/n_errors:.3f} [{fm2_lo:.3f}, {fm2_hi:.3f}]')

# As fraction of ALL questions (not just errors)
n_total = len(baseline)
n_correct = baseline['is_correct'].sum()
print(f'\nAs fraction of all {n_total} questions:')
print(f'  Correct: {n_correct/n_total:.3f}')
print(f'  FM1:     {n_fm1/n_total:.3f}')
print(f'  FM2:     {n_fm2/n_total:.3f}')
print(f'  Unclass: {n_unclassified/n_total:.3f}')

# Confidence distribution for FM1 vs FM2
classified = errors[errors['failure_mode'].isin(['FM1', 'FM2'])]
fm1_conf = classified[classified['failure_mode'] == 'FM1']['confidence'].dropna()
fm2_conf = classified[classified['failure_mode'] == 'FM2']['confidence'].dropna()
correct_conf = baseline[baseline['is_correct'] == True]['confidence'].dropna()

print(f'\nMean confidence:')
print(f'  Correct: {correct_conf.mean():.3f}')
print(f'  FM1:     {fm1_conf.mean():.3f}')
print(f'  FM2:     {fm2_conf.mean():.3f}')

if len(fm1_conf) > 0 and len(fm2_conf) > 0:
    u, p = stats.mannwhitneyu(fm1_conf, fm2_conf, alternative='two-sided')
    print(f'  Mann-Whitney FM1 vs FM2 confidence: p={p:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Stacked bar: outcome breakdown of all questions
ax = axes[0]
categories = ['All Questions']
bottom = 0
for val, lbl, color in [
    (n_correct/n_total, 'Correct', 'mediumseagreen'),
    (n_fm1/n_total,     'FM1 (Parametric Override)', 'tomato'),
    (n_fm2/n_total,     'FM2 (Reasoning Failure)', 'steelblue'),
    (n_unclassified/n_total, 'Unclassified Error', 'lightgrey'),
]:
    bar = ax.bar(categories, val, bottom=bottom, color=color, label=lbl,
                 edgecolor='white', linewidth=0.8)
    if val > 0.02:
        ax.text(0, bottom + val/2, f'{val:.1%}', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')
    bottom += val
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Outcome Breakdown\n(all 1,195 questions)')
ax.legend(fontsize=8, loc='upper right')

# FM split within errors only (with CI)
ax = axes[1]
fm_vals = [n_fm1/n_errors, n_fm2/n_errors]
fm_labels = ['FM1\nParametric\nOverride', 'FM2\nReasoning\nFailure']
fm_colors = ['tomato', 'steelblue']
fm_errs_lo = [n_fm1/n_errors - fm1_lo, n_fm2/n_errors - fm2_lo]
fm_errs_hi = [fm1_hi - n_fm1/n_errors, fm2_hi - n_fm2/n_errors]
bars = ax.bar(fm_labels, fm_vals, color=fm_colors, alpha=0.85, edgecolor='white')
ax.errorbar([0, 1], fm_vals, yerr=[fm_errs_lo, fm_errs_hi],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.set_ylabel('Proportion of Baseline Errors')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('FM Distribution Within Errors\n(95% Wilson CI)')

# Confidence distribution
ax = axes[2]
for data, lbl, color in [
    (correct_conf, f'Correct (μ={correct_conf.mean():.2f})', 'mediumseagreen'),
    (fm1_conf, f'FM1 (μ={fm1_conf.mean():.2f})', 'tomato'),
    (fm2_conf, f'FM2 (μ={fm2_conf.mean():.2f})', 'steelblue'),
]:
    if len(data) > 1:
        data.plot.kde(ax=ax, label=lbl, color=color, linewidth=2)
ax.set_xlabel('Model Confidence')
ax.set_title('Confidence Distribution\nby Outcome / FM Type')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

plt.suptitle('4.1.2 Baseline Failure Mode Distribution', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/fm_4_1_2_distribution.png', bbox_inches='tight')
plt.show()

## 4.1.3 Per-Subject Breakdown

In [ ]:
# Merge subject into errors
errors_subj = errors[errors['subject'].isin(SUBJECTS)].copy()

# Per-subject: overall accuracy, FM1 rate, FM2 rate (as % of errors in that subject)
subj_rows = []
for subj in SUBJECTS:
    subj_baseline = baseline[baseline['subject'] == subj]
    subj_errors   = errors_subj[errors_subj['subject'] == subj]
    n_subj = len(subj_baseline)
    n_err  = len(subj_errors)
    n_s_fm1 = (subj_errors['failure_mode'] == 'FM1').sum()
    n_s_fm2 = (subj_errors['failure_mode'] == 'FM2').sum()
    subj_rows.append({
        'Subject': subj,
        'N': n_subj,
        'Accuracy': subj_baseline['is_correct'].mean(),
        'Error Rate': n_err / n_subj,
        'FM1 (of errors)': n_s_fm1 / n_err if n_err > 0 else np.nan,
        'FM2 (of errors)': n_s_fm2 / n_err if n_err > 0 else np.nan,
        'FM1 count': n_s_fm1,
        'FM2 count': n_s_fm2,
        'FM1 (of N)': n_s_fm1 / n_subj,
        'FM2 (of N)': n_s_fm2 / n_subj,
    })

subj_df = pd.DataFrame(subj_rows).set_index('Subject')
display(subj_df[['N', 'Accuracy', 'Error Rate', 'FM1 (of errors)', 'FM2 (of errors)', 'FM1 count', 'FM2 count']]
        .style.format({'Accuracy': '{:.3f}', 'Error Rate': '{:.3f}',
                       'FM1 (of errors)': '{:.3f}', 'FM2 (of errors)': '{:.3f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: FM1 vs FM2 rate per subject (as % of all questions in subject)
ax = axes[0]
x = np.arange(len(subj_df))
w = 0.6
ax.bar(x, subj_df['FM1 (of N)'], w, label='FM1 (Parametric Override)', color='tomato', alpha=0.85)
ax.bar(x, subj_df['FM2 (of N)'], w, bottom=subj_df['FM1 (of N)'],
       label='FM2 (Reasoning Failure)', color='steelblue', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(subj_df.index, rotation=20, ha='right')
ax.set_ylabel('Proportion of Questions')
ax.set_title('FM1 + FM2 Error Rate per Subject\n(as % of all questions in subject)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()

# Heatmap: FM1 vs FM2 share within errors per subject
ax = axes[1]
hm_data = subj_df[['FM1 (of errors)', 'FM2 (of errors)']].rename(
    columns={'FM1 (of errors)': 'FM1', 'FM2 (of errors)': 'FM2'})
sns.heatmap(hm_data, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Proportion of Subject Errors'})
ax.set_title('FM Share Within Errors per Subject')
ax.set_xlabel('')

plt.suptitle('4.1.3 Per-Subject Failure Mode Breakdown', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/fm_4_1_3_per_subject.png', bbox_inches='tight')
plt.show()

## 4.1.4 Confidence × Failure Mode

Are FM1 (Parametric Override) errors more confident than FM2 (Reasoning Failure)? This would indicate the model "knows" wrong law more confidently than it fails to reason.

In [ ]:
classified = errors[errors['failure_mode'].isin(['FM1', 'FM2'])]
fm1_conf = classified[classified['failure_mode'] == 'FM1']['confidence'].dropna()
fm2_conf = classified[classified['failure_mode'] == 'FM2']['confidence'].dropna()
correct_conf = baseline[baseline['is_correct'] == True]['confidence'].dropna()

print('Confidence stats by group:')
for label, data in [('Correct', correct_conf), ('FM1', fm1_conf), ('FM2', fm2_conf)]:
    print(f'  {label}: mean={data.mean():.3f}, median={data.median():.3f}, std={data.std():.3f}, n={len(data)}')

# Statistical tests
u_fm, p_fm = stats.mannwhitneyu(fm1_conf, fm2_conf, alternative='two-sided')
u_c1, p_c1 = stats.mannwhitneyu(correct_conf, fm1_conf, alternative='greater')
u_c2, p_c2 = stats.mannwhitneyu(correct_conf, fm2_conf, alternative='greater')
print(f'\nMann-Whitney (two-sided):')
print(f'  FM1 vs FM2:     p={p_fm:.4f}')
print(f'  Correct > FM1:  p={p_c1:.4f}')
print(f'  Correct > FM2:  p={p_c2:.4f}')

# Proportion of high-confidence errors (>0.9)
thresh = 0.9
for label, data in [('FM1', fm1_conf), ('FM2', fm2_conf)]:
    hi = (data > thresh).mean()
    print(f'  {label} errors with confidence > {thresh}: {hi:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# KDE
ax = axes[0]
for data, lbl, color in [
    (correct_conf, f'Correct (μ={correct_conf.mean():.2f})', 'mediumseagreen'),
    (fm1_conf, f'FM1 (μ={fm1_conf.mean():.2f})', 'tomato'),
    (fm2_conf, f'FM2 (μ={fm2_conf.mean():.2f})', 'steelblue'),
]:
    data.plot.kde(ax=ax, label=lbl, color=color, linewidth=2)
ax.set_xlabel('Model Confidence')
ax.set_title('Confidence KDE by FM Type')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

# Box
ax = axes[1]
bp = ax.boxplot([correct_conf, fm1_conf, fm2_conf],
                labels=['Correct', 'FM1', 'FM2'],
                patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
for patch, c in zip(bp['boxes'], ['mediumseagreen', 'tomato', 'steelblue']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.set_ylabel('Confidence')
ax.set_title('Confidence Distribution\n(boxplot)')

# CDF — high-confidence error rate
ax = axes[2]
for data, lbl, color in [
    (fm1_conf, 'FM1', 'tomato'),
    (fm2_conf, 'FM2', 'steelblue'),
]:
    sorted_data = np.sort(data)
    cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
    ax.plot(sorted_data, cdf, color=color, label=lbl, linewidth=2)
ax.axvline(0.9, color='black', linestyle='--', linewidth=1, alpha=0.6, label='threshold=0.9')
ax.set_xlabel('Confidence')
ax.set_ylabel('CDF')
ax.set_title('Cumulative Confidence Distribution\n(FM1 vs FM2 errors)')
ax.legend(fontsize=8)

plt.suptitle('4.1.4 Confidence × Failure Mode', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/fm_4_1_4_confidence.png', bbox_inches='tight')
plt.show()

## 4.1.5 Justification Length × Failure Mode

Are FM2 (Reasoning Failure) errors associated with longer justifications? Longer reasoning ≠ correct reasoning — supports the 'steps skipped / shallow' finding from the earlier manual taxonomy.

In [ ]:
# Justification length from the FM classifier output (proxy for reasoning depth)
classified = errors[errors['failure_mode'].isin(['FM1', 'FM2'])].copy()
classified['justification_len'] = classified['justification'].fillna('').str.len()

# Also use raw model response length from baseline
baseline['response_len'] = baseline['raw_response_1'].fillna('').str.len()
errors_resp = errors.merge(baseline[['idx', 'response_len']], on='idx', how='left')
errors_resp = errors_resp[errors_resp['failure_mode'].isin(['FM1', 'FM2'])]

fm1_resp = errors_resp[errors_resp['failure_mode'] == 'FM1']['response_len'].dropna()
fm2_resp = errors_resp[errors_resp['failure_mode'] == 'FM2']['response_len'].dropna()
correct_resp = baseline[baseline['is_correct'] == True]['response_len'].dropna()

print('Model response length (chars) by group:')
for label, data in [('Correct', correct_resp), ('FM1', fm1_resp), ('FM2', fm2_resp)]:
    print(f'  {label}: mean={data.mean():.0f}, median={data.median():.0f}, n={len(data)}')

u, p = stats.mannwhitneyu(fm1_resp, fm2_resp, alternative='two-sided')
print(f'\nMann-Whitney FM1 vs FM2 response length: p={p:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Response length box
ax = axes[0]
bp = ax.boxplot([correct_resp, fm1_resp, fm2_resp],
                labels=['Correct', 'FM1', 'FM2'],
                patch_artist=True, medianprops={'color': 'black', 'linewidth': 2},
                showfliers=False)
for patch, c in zip(bp['boxes'], ['mediumseagreen', 'tomato', 'steelblue']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.set_ylabel('Response Length (chars)')
ax.set_title('Model Response Length by Outcome / FM Type\n(outliers hidden)')

# Response length KDE
ax = axes[1]
for data, lbl, color in [
    (correct_resp, f'Correct (μ={correct_resp.mean():.0f})', 'mediumseagreen'),
    (fm1_resp, f'FM1 (μ={fm1_resp.mean():.0f})', 'tomato'),
    (fm2_resp, f'FM2 (μ={fm2_resp.mean():.0f})', 'steelblue'),
]:
    data.plot.kde(ax=ax, label=lbl, color=color, linewidth=2)
ax.set_xlabel('Response Length (chars)')
ax.set_title('Response Length KDE')
ax.legend(fontsize=8)

plt.suptitle('4.1.5 Justification/Response Length × Failure Mode', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/fm_4_1_5_response_length.png', bbox_inches='tight')
plt.show()